# Лабораторная работа 9. Метод главных компонент, SVD и многомерное шкалирование

**Курс «Машинное обучение», 4 курс, каф. ФН1**

| | |
|---|---|
| Место в курсе | после лекции 8, завершает курс |
| Опора на лекции | лекция 8: выборочная ковариационная матрица (опр. 8.1), первая главная компонента (теорема 8.2), полная дисперсия (утв. 8.5), доля объяснённой дисперсии, численный пример 8.7, сингулярное разложение (теорема 8.9), многомерное шкалирование |
| Трудоёмкость | 2 ч аудиторно + домашняя работа (3–4 ч) |

## Цель работы

Разобраться, что именно ищет PCA и почему ответ зависит от единиц измерения; увидеть теорему Эккарта–Янга в действии на сжатии изображений; построить карту объектов по одним лишь попарным расстояниям. Последняя часть — сводная по всему курсу.

## Как устроено занятие

Ноутбук разбирается в аудитории: код запускается и обсуждается по ходу.
Большая часть ячеек уже написана — их нужно **прочитать и запустить**,
разобравшись, что происходит и почему.

Ячейки, помеченные `# ✍ ЗАДАНИЕ НА СЕМИНАРЕ`, заполняются самостоятельно
прямо на занятии; их немного, и каждая занимает несколько строк. Ячейки
**Вывод** — тоже ваши: короткий ответ своими словами на поставленный вопрос.

Дома выполняется отдельный ноутбук `lab09_homework.ipynb` — там
задания крупнее и делать их нужно самому.

> **Индивидуальный вариант.** Датасет и набор методов выдаются по вашему ФИО
> (см. ячейку ниже) — и на семинаре, и в домашней работе.

In [ ]:
# Служебная ячейка: импорты, стиль графиков, воспроизводимость.
import sys, pathlib, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# Модули практикума (variants.py, labdata.py) ищем рядом с ноутбуком.
# Если их нет -- значит, ноутбук открыт в Colab: скачиваем из репозитория курса.
COURSE_FILES_URL = "https://raw.githubusercontent.com/sharipovaka/mltest1/main/notebooks"


def _course_modules_dir():
    here = pathlib.Path.cwd()
    for parent in [here, *here.parents][:4]:
        if (parent / "variants.py").exists():
            return str(parent)
    import urllib.request
    for name in ("variants.py", "labdata.py"):
        if not pathlib.Path(name).exists():
            urllib.request.urlretrieve(f"{COURSE_FILES_URL}/{name}", name)
            print(f"загружен {name} из репозитория курса")
    return str(here)


sys.path.insert(0, _course_modules_dir())
from variants import get_variant, describe_variant  # noqa: E402

RANDOM_STATE = 42          # единый seed на всю работу: результаты воспроизводимы
rng = np.random.default_rng(RANDOM_STATE)

plt.rcParams.update({
    "figure.figsize": (7.5, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

print("numpy", np.__version__, "| pandas", pd.__version__)

from sklearn.datasets import load_digits, load_iris, load_wine
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from labdata import load_personal

## Индивидуальный вариант

Впишите своё ФИО (или почту) в переменную `STUDENT` — вариант вычисляется детерминированно, при повторном запуске он тот же самый.

In [ ]:
STUDENT = "Фамилия Имя Отчество"   # <-- впишите себя

variant = get_variant(STUDENT, lab=9)
describe_variant(variant)

---
# Часть 1. Что ищет PCA

Определение 8.1: выборочная ковариационная матрица центрированных данных
$C = \frac1\ell X^{\mathsf T}X$. Теорема 8.2: первая главная компонента — это
собственный вектор $C$ с наибольшим собственным числом, а дисперсия проекции
равна этому собственному числу.

Начнём с точного воспроизведения примера 8.7 из конспекта: четыре центрированные
точки $(2,1)$, $(-1,-2)$, $(1,2)$, $(-2,-1)$. Там получено $\lambda = 4.5$ и
$0.5$, $w_1 = (1,1)/\sqrt2$, доля первой компоненты 90 %.

In [ ]:
X_ex = np.array([[2.0, 1.0], [-1.0, -2.0], [1.0, 2.0], [-2.0, -1.0]])
print("центр выборки:", X_ex.mean(axis=0), "(нулевой -- данные уже центрированы)")

C = X_ex.T @ X_ex / len(X_ex)                     # опр. 8.1
print("C =\n", C)
print(f"след Tr C = {np.trace(C)}, определитель = {np.linalg.det(C):.4f}")

In [ ]:
# ✍ ЗАДАНИЕ НА СЕМИНАРЕ — допишите эту ячейку

# TODO (3-4 строки): найдите собственные пары C через np.linalg.eigh,
#   отсортируйте по УБЫВАНИЮ собственного числа и выведите:
#   собственные числа, первую компоненту и долю дисперсии lam[0]/sum(lam).
#   Сверьте с конспектом: 4.5 и 0.5, (1,1)/sqrt(2), доля 0.9.

In [ ]:
# Утверждение 8.5: сумма собственных чисел = след C = полная дисперсия
print(f"сумма собственных чисел = {lam.sum():.4f}, Tr C = {np.trace(C):.4f}")

# Проекция на первую компоненту и восстановление
w1 = W[:, 0] * np.sign(W[0, 0])
z = X_ex @ w1
X_rec = np.outer(z, w1)
print(f"\nпроекции: {np.round(z, 4)}   (в конспекте +-3/sqrt(2) = +-{3 / np.sqrt(2):.4f})")
print(f"восстановление x_1: {np.round(X_rec[0], 4)}   (в конспекте (1.5, 1.5))")
print(f"суммарная ошибка: {((X_ex - X_rec) ** 2).sum():.4f}"
      f"   (в конспекте lambda_2 * l = 0.5 * 4 = 2)")

> **Вывод.** Совпало ли с конспектом? Почему ошибка восстановления равна в точности $\lambda_2\ell$?
>
> *(ваш ответ здесь)*

---
# Часть 2. Масштабирование меняет ответ

PCA ищет направления максимальной дисперсии, а дисперсия зависит от единиц
измерения. Признак «доход в рублях» имеет дисперсию в миллиарды раз большую, чем
«доля клиентов», и первая компонента совпадёт с ним почти точно — независимо от
содержательной важности.

In [ ]:
X_w, y_w = load_wine(return_X_y=True)
feat = np.array(load_wine().feature_names)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
for ax, (name, Xp) in zip(axes, [("без масштабирования", X_w),
                                 ("StandardScaler", StandardScaler().fit_transform(X_w))]):
    p = PCA(n_components=2).fit(Xp)
    Z = p.transform(Xp)
    for c, mk in zip(range(3), "os^"):
        ax.scatter(*Z[y_w == c].T, s=18, marker=mk, label=f"класс {c}")
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.legend(fontsize=8)
    ax.set_title(f"{name}\nдоля дисперсии PC1 = {p.explained_variance_ratio_[0]:.3f}",
                 fontsize=10)
    top = np.abs(p.components_[0]).argsort()[::-1][:3]
    print(f"{name}: PC1 объясняет {p.explained_variance_ratio_[0]:.1%}; главные вклады:",
          ", ".join(f"{feat[t]} ({p.components_[0][t]:+.3f})" for t in top))
plt.tight_layout(); plt.show()

print("\nстандартные отклонения признаков:")
print(pd.Series(X_w.std(axis=0), index=feat).sort_values(ascending=False).head(3).round(2))

> **Вывод.** Какой признак «захватил» первую компоненту без масштабирования? Всегда ли нужно стандартизовать перед PCA?
>
> *(ваш ответ здесь)*

---
# Часть 3. Сжатие: теорема Эккарта–Янга

Следствие из теоремы 8.9: усечённое SVD ранга $k$ — **наилучшее** приближение
матрицы рангом $k$ в норме Фробениуса, причём

$$
\min_{\mathrm{rank}(B)\le k}\|X - B\|_F^2 \;=\; \sum_{j>k}\sigma_j^2 ,
$$

то есть ошибка равна сумме квадратов **отброшенных** сингулярных чисел.

> **Напоминание — норма Фробениуса и ранг.** $\|A\|_F^2 = \sum_{i,j}a_{ij}^2$ — просто сумма квадратов всех элементов,
> то есть обычная евклидова норма, если вытянуть матрицу в один вектор.
> В NumPy: `(A ** 2).sum()` или `np.linalg.norm(A, "fro") ** 2`.
>
> **Ранг** $B$ не превосходит $k$ означает, что все строки $B$ лежат в некотором
> $k$-мерном подпространстве. Так что теорема Эккарта–Янга говорит буквально
> следующее: среди всех способов уложить данные в $k$ измерений наилучший в
> смысле суммы квадратов ошибок — это усечённое SVD. Причём ошибка известна
> заранее, не надо ничего перебирать: $\sum_{j>k}\sigma_j^2$.

In [ ]:
digits = load_digits()
X_dig = digits.data                        # 1797 изображений 8x8, значения 0..16
Xc = X_dig - X_dig.mean(axis=0)
U, s, Vt = np.linalg.svd(Xc, full_matrices=False)

print(" k   ||X - X_k||_F^2   сумма отброшенных sigma^2   расхождение")
for k in [1, 5, 10, 20, 40]:
    Xk = U[:, :k] @ np.diag(s[:k]) @ Vt[:k] + X_dig.mean(axis=0)
    lhs, rhs = ((X_dig - Xk) ** 2).sum(), (s[k:] ** 2).sum()
    print(f"{k:3d}   {lhs:14.4f}   {rhs:24.4f}   {abs(lhs - rhs):.2e}")

In [ ]:
ks = [1, 2, 4, 8, 16, 32, 64]
fig, axes = plt.subplots(3, len(ks) + 1, figsize=(1.5 * (len(ks) + 1), 4.8))
for r, i in enumerate([0, 1, 2]):
    axes[r, 0].imshow(X_dig[i].reshape(8, 8), cmap="gray_r")
    axes[r, 0].set_ylabel(f"цифра {digits.target[i]}", fontsize=8)
    axes[r, 0].set_xticks([]); axes[r, 0].set_yticks([])
    if r == 0:
        axes[r, 0].set_title("оригинал", fontsize=8)
for c, k in enumerate(ks, start=1):
    p = PCA(n_components=k).fit(X_dig)
    rec = p.inverse_transform(p.transform(X_dig))
    for r, i in enumerate([0, 1, 2]):
        axes[r, c].imshow(rec[i].reshape(8, 8), cmap="gray_r")
        axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
        if r == 0:
            axes[r, c].set_title(f"$k={k}$\n{p.explained_variance_ratio_.sum():.0%}",
                                 fontsize=8)
plt.tight_layout(); plt.show()

cum_dig = np.cumsum(PCA().fit(X_dig).explained_variance_ratio_)
print(f"для 90% дисперсии достаточно {int(np.searchsorted(cum_dig, 0.90) + 1)} компонент из 64")

In [ ]:
# «Собственные цифры»: первые компоненты как изображения
p19 = PCA(n_components=19).fit(X_dig)
fig, axes = plt.subplots(2, 10, figsize=(14, 3.2))
axes[0, 0].imshow(p19.mean_.reshape(8, 8), cmap="gray_r")
axes[0, 0].set_title("среднее", fontsize=8)
for j, ax in enumerate(np.ravel(axes)[1:20]):
    ax.imshow(p19.components_[j].reshape(8, 8), cmap="RdBu_r")
    ax.set_title(f"PC{j + 1}\n{p19.explained_variance_ratio_[j]:.1%}", fontsize=7)
for ax in np.ravel(axes):
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("Главные компоненты изображений")
plt.tight_layout(); plt.show()

> **Вывод.** Совпала ли ошибка восстановления с суммой отброшенных $\sigma_j^2$? При каком $k$ цифры узнаваемы и что изображают первые компоненты?
>
> *(ваш ответ здесь)*

---
# Часть 4. PCA как предобработка

Снижение размерности перед обучением — компромисс: теряем часть информации,
но уменьшаем разброс (занятие 5) и ускоряем обучение. Важно, что PCA
**не использует метки** — он максимизирует дисперсию, а не разделимость классов.

И конечно, PCA должен быть шагом `Pipeline`: иначе главные компоненты считаются
по всей выборке, включая контрольную, — это утечка из занятия 5.

In [ ]:
X_d, y_d = load_digits(return_X_y=True)
cv9 = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)

rows = []
for k in [2, 5, 10, 20, 30, 40, 64]:
    pipe = Pipeline([("scale", StandardScaler()), ("pca", PCA(n_components=k)),
                     ("clf", LogisticRegression(max_iter=3000))])
    rows.append({"компонент": k, "точность": cross_val_score(pipe, X_d, y_d, cv=cv9).mean()})
base = cross_val_score(Pipeline([("scale", StandardScaler()),
                                 ("clf", LogisticRegression(max_iter=3000))]),
                       X_d, y_d, cv=cv9).mean()
display(pd.DataFrame(rows).set_index("компонент").round(4))
print(f"без PCA (все 64 признака): {base:.4f}")

In [ ]:
# ✍ ЗАДАНИЕ НА СЕМИНАРЕ — допишите эту ячейку

from sklearn.feature_selection import SelectKBest, f_classif

warnings.filterwarnings("ignore")     # у digits есть всегда-чёрные пиксели: f_classif делит на 0

# TODO (3-4 строки): для k in [2, 5, 10, 20, 30] сравните два конвейера
#   при одинаковом числе признаков:
#     StandardScaler -> PCA(k)                    -> LogisticRegression
#     StandardScaler -> SelectKBest(f_classif, k) -> LogisticRegression
#   Оценивайте cross_val_score(..., cv=cv9). Кто лучше при малом k?

> **Вывод.** Сколько компонент достаточно, чтобы не потерять качество? Кто лучше при малом $k$ — PCA или отбор признаков, и почему?
>
> *(ваш ответ здесь)*

---
# Часть 5. Карта по одним лишь расстояниям

Иногда координат нет вовсе — есть только матрица попарных несходств: экспертные
оценки, генетические дистанции, время в пути. Многомерное шкалирование
восстанавливает по ней координаты.

Классическое MDS (§5 лекции 8): по матрице квадратов расстояний двойным
центрированием восстанавливается матрица Грама
$B = -\frac12 J D^{(2)} J$, где $J = I - \frac1\ell\mathbf1\mathbf1^{\mathsf T}$,
а координаты берутся из $k$ старших собственных пар $B$.

> **Напоминание — матрица Грама и диаграмма Шепарда.** **Матрица Грама** — матрица попарных скалярных произведений,
> $B_{ij} = \langle x_i, x_j\rangle$. Ключ ко всему методу — тождество
> $\|x_i-x_j\|^2 = B_{ii} - 2B_{ij} + B_{jj}$: расстояния выражаются через
> скалярные произведения, и наоборот — если зафиксировать начало координат в
> центре выборки, то по расстояниям однозначно восстанавливаются скалярные
> произведения. Это и делает двойное центрирование. Дальше $B = ZZ^{\mathsf T}$
> даёт координаты $Z$ — с точностью до поворота и отражения, потому что они
> скалярных произведений не меняют.
>
> **Диаграмма Шепарда** — проверка качества вложения: по горизонтали исходные
> несходства, по вертикали расстояния на построенной карте, по одной точке на
> пару объектов. Идеальному вложению отвечает биссектриса; систематический изгиб
> означает, что размерности не хватило.

In [ ]:
cities = ["Калининград", "Санкт-Петербург", "Москва", "Ростов-на-Дону",
          "Казань", "Екатеринбург", "Новосибирск", "Иркутск", "Владивосток"]
D_km = np.array([
    [   0,  800, 1090, 1650, 1780, 2380, 3760, 5350, 7900],
    [ 800,    0,  635, 1520, 1330, 1780, 3110, 4700, 6400],
    [1090,  635,    0,  960,  720, 1420, 2810, 4200, 6430],
    [1650, 1520,  960,    0,  980, 1750, 3100, 4500, 6800],
    [1780, 1330,  720,  980,    0,  730, 2100, 3500, 5750],
    [2380, 1780, 1420, 1750,  730,    0, 1400, 2800, 5050],
    [3760, 3110, 2810, 3100, 2100, 1400,    0, 1450, 3700],
    [5350, 4700, 4200, 4500, 3500, 2800, 1450,    0, 2500],
    [7900, 6400, 6430, 6800, 5750, 5050, 3700, 2500,    0]], dtype=float)
print("координат нет -- только попарные расстояния по прямой, км")

In [ ]:
# ✍ ЗАДАНИЕ НА СЕМИНАРЕ — допишите эту ячейку

def classic_mds(D, k=2):
    """Классическое MDS.

    1) J = I - (1/l) * матрица из единиц;
    2) B = -1/2 * J @ D^2 @ J   (двойное центрирование);
    3) собственные пары B (np.linalg.eigh), взять k старших;
    4) координаты = U_k * sqrt(lambda_k), отрицательные lambda обнулить.
    Возвращает (координаты, спектр по убыванию).
    """
    raise NotImplementedError


# TODO: восстановите координаты по D_km, посчитайте среднюю ошибку
#       воспроизведения расстояний и долю первых двух собственных чисел.

In [ ]:
xs, ys = coords[:, 0], coords[:, 1]
if xs[cities.index("Владивосток")] < xs[cities.index("Калининград")]:
    xs = -xs                                    # разворот: восток направо

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.scatter(xs, ys, s=60, color="#128C7E")
for nm, a, b in zip(cities, xs, ys):
    ax1.annotate(nm, (a, b), fontsize=8, xytext=(4, 4), textcoords="offset points")
ax1.set_xlabel("MDS-1"); ax1.set_ylabel("MDS-2"); ax1.set_aspect("equal")
ax1.set_title("Карта, восстановленная по одним лишь расстояниям")

ax2.scatter(D_km[iu], D_rec[iu], s=28, color="#C97A2B")
ax2.plot([0, D_km.max()], [0, D_km.max()], "k--", lw=1)
ax2.set_xlabel("истинное расстояние, км"); ax2.set_ylabel("расстояние на карте")
ax2.set_title("Диаграмма Шепарда")
plt.tight_layout(); plt.show()

In [ ]:
# При евклидовой метрике классическое MDS и PCA -- одно и то же вычисление
X_i = StandardScaler().fit_transform(load_iris(return_X_y=True)[0])
sq = (X_i ** 2).sum(axis=1)
D_i = np.sqrt(np.maximum(sq[:, None] - 2 * X_i @ X_i.T + sq[None, :], 0))

Z_pca = PCA(n_components=2).fit_transform(X_i)
Z_mds, _ = classic_mds(D_i, k=2)
dist = lambda Z: np.sqrt(np.maximum(((Z[:, None] - Z[None]) ** 2).sum(-1), 0))
print(f"max|расстояния PCA - расстояния MDS| = {np.abs(dist(Z_pca) - dist(Z_mds)).max():.2e}")
print("=> одна и та же конфигурация (с точностью до поворота и отражения)")

> **Вывод.** Насколько точно восстановилась карта? Почему MDS и PCA совпали и в какой ситуации MDS применим, а PCA — нет?
>
> *(ваш ответ здесь)*

---
# Часть 6. Сводная: полный цикл на своей выборке

Заключительная часть курса. Проходим весь путь: предобработка (занятие 1),
снижение размерности, сравнение моделей из разных семейств с подбором
гиперпараметров по скользящему контролю (занятие 5), честная финальная оценка
и интерпретация.

> **Напоминание — scree-plot и правило локтя.** **Scree-plot** («осыпь») — график собственных чисел $\lambda_j$ по убыванию
> номера компоненты. Название геологическое: крутой склон из крупных компонент
> и пологая осыпь мелочи у подножия. **Правило локтя** — взять $k$ в точке
> излома, где крутое падение сменяется пологим: считается, что дальше идёт шум.
>
> Правило неформальное — излома может не быть вовсе, и тогда решают по доле
> объяснённой дисперсии или, что честнее, по качеству итоговой модели (это
> домашняя работа). Численно излом ищут как максимум второй разности
> $\lambda_{j-1} - 2\lambda_j + \lambda_{j+1}$.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import KNeighborsClassifier

data = load_personal(variant)
Xtr, Xte, ytr, yte = data["X_train"], data["X_test"], data["y_train"], data["y_test"]
if data["task"] == "regression":
    thr = np.median(ytr)
    ytr, yte = (ytr > thr).astype(int), (yte > thr).astype(int)
ytr, yte = np.asarray(ytr).astype(int), np.asarray(yte).astype(int)

p_full = PCA().fit(Xtr)
cum = np.cumsum(p_full.explained_variance_ratio_)
rule = variant["n_components_rule"]
if "95" in rule:
    k_var = int(np.searchsorted(cum, 0.95) + 1)
elif "90" in rule:
    k_var = int(np.searchsorted(cum, 0.90) + 1)
else:                                    # правило локтя: максимум второй разности
    k_var = int(np.argmax(np.diff(p_full.explained_variance_, 2)) + 2)
print(f"признаков после предобработки: {Xtr.shape[1]}")
print(f"ваше правило «{rule}» -> k = {k_var}")

In [ ]:
Z2 = PCA(n_components=2).fit_transform(Xtr)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.6))
for c, mk in zip([0, 1], "os"):
    ax1.scatter(*Z2[ytr == c].T, s=14, marker=mk, alpha=0.6, label=f"класс {c}")
ax1.set_xlabel("PC1"); ax1.set_ylabel("PC2"); ax1.legend(fontsize=8)
ax1.set_title("Выборка в первых двух главных компонентах")
ax2.plot(np.arange(1, len(cum) + 1), cum, "o-", lw=2)
ax2.axvline(k_var, ls="--", color="#C97A2B", label=f"ваше правило: $k = {k_var}$")
ax2.set_xlabel("число компонент"); ax2.set_ylabel("накопленная доля дисперсии")
ax2.legend(fontsize=8); ax2.set_title("Выбор числа компонент")
plt.tight_layout(); plt.show()

In [ ]:
cvf = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
candidates = {
    "линейная (логистическая регрессия)":
        (Pipeline([("pca", PCA()), ("clf", LogisticRegression(max_iter=5000))]),
         {"pca__n_components": [k_var, Xtr.shape[1]], "clf__C": np.logspace(-2, 2, 5)}),
    "метрическая (kNN)":
        (Pipeline([("pca", PCA()), ("clf", KNeighborsClassifier())]),
         {"pca__n_components": [k_var, Xtr.shape[1]], "clf__n_neighbors": [3, 7, 15, 31]}),
    "ансамбль (случайный лес)":
        (Pipeline([("clf", RandomForestClassifier(random_state=RANDOM_STATE))]),
         {"clf__n_estimators": [200], "clf__max_features": ["sqrt", 0.5],
          "clf__max_depth": [5, 12, None]}),
}

rows, fitted = [], {}
for nm, (pipe, grid) in candidates.items():
    gs = GridSearchCV(pipe, grid, cv=cvf, scoring="roc_auc", n_jobs=-1).fit(Xtr, ytr)
    fitted[nm] = gs.best_estimator_
    rows.append({"модель": nm, "CV AUC": gs.best_score_,
                 "AUC на контроле": roc_auc_score(yte, gs.predict_proba(Xte)[:, 1])})
    print(f"{nm}: {gs.best_params_}")
rows.append({"модель": "константа", "CV AUC": 0.5, "AUC на контроле": 0.5})
final = pd.DataFrame(rows).set_index("модель")
display(final.round(4))

In [ ]:
from sklearn.inspection import permutation_importance

best_name = final.drop(index="константа")["CV AUC"].idxmax()
perm = permutation_importance(fitted[best_name], Xte, yte, n_repeats=15,
                              random_state=0, scoring="roc_auc")
imp = pd.Series(perm.importances_mean, index=data["feature_names"]).nlargest(10)

fig, ax = plt.subplots(figsize=(8, 4.5))
imp[::-1].plot.barh(ax=ax, color="#128C7E")
ax.set_xlabel("падение ROC-AUC при перестановке признака")
ax.set_title(f"Что определяет предсказание: {best_name}")
plt.tight_layout(); plt.show()

> **Вывод.** Какая модель победила и насколько уверенно? Помог ли PCA? Согласуется ли важность признаков с предметной областью?
>
> *(ваш ответ здесь)*

## Итоги занятия

Ответьте письменно на контрольные вопросы (по 2–4 предложения):

1. Первая главная компонента — собственный вектор $C$ с наибольшим собственным числом. Почему именно наибольшим? Изложите рассуждение теоремы 8.2 своими словами.
2. PCA не использует метки классов. Постройте пример двух классов в $\mathbb{R}^2$, для которых проекция на первую главную компоненту полностью уничтожает разделимость.
3. У вас есть только матрица попарных несходств между 200 объектами, координат нет. Какие методы курса вы можете применить, а какие — нет?
4. Оглядываясь на весь курс: назовите три проверки, которые вы теперь будете делать всегда, прежде чем поверить в качество модели.

---

**Дома:** откройте `lab09_homework.ipynb` — там две задачи: PCA двумя способами (и почему на практике всегда SVD) и выбор числа компонент по качеству, а не по дисперсии.